# Chapter 7 — A stellar stream: the tidal tails of Pal 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/07_pal5_stream.ipynb)

**Goal.** A satellite is a compact overdensity; a disrupted satellite is spread along its orbit into a long, thin
stream. The globular cluster Pal 5 is the classic example: SDSS revealed its two tidal tails (Odenkirchen et al.
2001, https://ui.adsabs.harvard.edu/abs/2001ApJ...548L.165O; Rockosi et al. 2002,
https://ui.adsabs.harvard.edu/abs/2002AJ....124..349R). The same matched filter as for satellites reveals it
(Chapter 4); what changes is the geometry. We rotate to coordinates aligned with the stream, measure its track
and width along it, and judge the significance of each segment against control strips.

**You will learn**
1. to build a great-circle coordinate frame $(\phi_1, \phi_2)$ through a point and along a direction;
2. to measure a stream's track, width and density in bins of $\phi_1$ with a Poisson likelihood that includes the
   survey footprint;
3. to calibrate the significance of stream segments with control strips;
4. to compare the measured track with a published one (`galstreams`, Mateu 2023).

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.optimize import minimize

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)


lvdb = load('lvdb').set_index('key')
iso = load('isochrones_sdss')
p5 = lvdb.loc['palomar_05']

## 7.1 Selecting stars like Pal 5's

Pal 5 is ~11–12 Gyr old with [Fe/H] ≈ −1.3; its distance modulus is 16.71 (LVDB). We build a CMD mask exactly as in
Chapter 5 (cells ranked by $s/b$, keeping at least half of the cluster's stars), with $b$ from the whole field.

In [ ]:
cb, mb = np.arange(-0.3, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')


def satellite_cmd(mu, logage, feh, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    """Normalised CMD density of a simple stellar population (as in Chapters 4-5)."""
    I = iso[np.isclose(iso.logage, logage) & np.isclose(iso.feh, feh) & (iso.stage <= 7)].sort_values('m_ini')
    c, g = (I.g - I.r).to_numpy(), I.g.to_numpy()
    arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(c), np.diff(g)))])
    s = np.linspace(0, arc[-1], 3000)
    pc, pg, w = np.interp(s, arc, c), np.interp(s, arc, g) + mu, np.gradient(np.interp(s, arc, I.int_imf))
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    S = np.einsum('k,kij->ij', w, np.exp(-0.5 * ((CC - pc[:, None, None]) / sc[:, None, None])**2
                                         - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_mask(S, B, keep=0.5):
    order = np.argsort((S / B).ravel())[::-1]
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])
    M = np.zeros(S.size, bool)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


cat = load('pal5_sdss')
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
M = cmd_mask(satellite_cmd(p5.distance_modulus, 10.05, -1.3), B)
i, j = np.digitize(c0, cb) - 1, np.digitize(g0, mb) - 1
sel = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm)) & M[np.clip(i, 0, len(cc) - 1), np.clip(j, 0, len(mm) - 1)]
ra, dec = cat.ra.to_numpy()[sel], cat.dec.to_numpy()[sel]
print(f'{sel.sum():,} of {len(cat):,} stars in the CMD mask')

## 7.2 A coordinate frame aligned with the stream

Let $\hat v_0$ be the unit vector to Pal 5, and $\hat e$, $\hat n$ the local east and north directions there. A
great circle through Pal 5 at position angle $\alpha$ (north through east) has direction $\hat d = \sin\alpha\,\hat e +
\cos\alpha\,\hat n$ and pole $\hat p = \hat v_0 \times \hat d$. The rotation matrix with rows $(\hat v_0, \hat d, \hat p)$
maps a sky unit vector $\hat v$ to $\hat v' = R\hat v$, and

$$ \phi_1 = {\rm atan2}(v'_y, v'_x), \qquad \phi_2 = \arcsin v'_z ,$$

so Pal 5 is at (0, 0) and the great circle is $\phi_2 = 0$. The inverse is $\hat v = R^{\sf T}\hat v'$. We choose $\alpha$
by maximising the number of masked stars within 0.25° of the great circle at $1.5° < |\phi_1| < 6°$ (avoiding the cluster).

In [ ]:
def unit(ra, dec):
    ra, dec = np.radians(ra), np.radians(dec)
    return np.array([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])


def frame(ra0, dec0, alpha):
    """Rotation matrix to stream coordinates through (ra0, dec0) along position angle alpha (deg)."""
    v0 = unit(ra0, dec0)
    a0, d0 = np.radians(ra0), np.radians(dec0)
    e = np.array([-np.sin(a0), np.cos(a0), 0])
    n = np.array([-np.sin(d0) * np.cos(a0), -np.sin(d0) * np.sin(a0), np.cos(d0)])
    d = np.sin(np.radians(alpha)) * e + np.cos(np.radians(alpha)) * n
    return np.array([v0, d, np.cross(v0, d)])


def to_stream(R, ra, dec):
    v = R @ unit(ra, dec)
    return np.degrees(np.arctan2(v[1], v[0])), np.degrees(np.arcsin(np.clip(v[2], -1, 1)))


def from_stream(R, phi1, phi2):
    v = R.T @ unit(phi1, phi2)
    return np.degrees(np.arctan2(v[1], v[0])) % 360, np.degrees(np.arcsin(np.clip(v[2], -1, 1)))


alphas = np.arange(0, 180, 1.0)
score = []
for a in alphas:
    f1, f2 = to_stream(frame(p5.ra, p5.dec, a), ra, dec)
    score.append(np.sum((np.abs(f2) < 0.25) & (np.abs(f1) > 1.5) & (np.abs(f1) < 6)))
alpha = alphas[np.argmax(score)]
R = frame(p5.ra, p5.dec, alpha)
phi1, phi2 = to_stream(R, ra, dec)

# round trip on random points: the inverse transformation must recover them
rng = np.random.default_rng(7)
tra, tdec = rng.uniform(222, 244, 1000), rng.uniform(-8, 8, 1000)
bra, bdec = from_stream(R, *to_stream(R, tra, tdec))
rt = np.max(np.hypot((bra - tra) * np.cos(np.radians(tdec)), bdec - tdec))
print(f'best position angle alpha = {alpha:.0f} deg; round-trip error {rt:.1e} deg')
assert rt < 1e-6

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(alphas, score)
ax[0].set(xlabel='position angle α of the great circle [deg]', ylabel='masked stars near the circle')
h = ax[1].hist2d(phi1, phi2, [np.arange(-12, 12.01, 0.1), np.arange(-4, 4.01, 0.1)], cmap='Greys', vmax=12)
ax[1].set(xlabel=r'$\phi_1$ [deg]', ylabel=r'$\phi_2$ [deg]', title='masked stars in the stream frame', aspect='equal')
plt.tight_layout()

**Figure 7.1.** Left: the number of masked stars near a great circle through Pal 5, as a function of its
orientation; the tails produce a sharp maximum. Right: the masked stars in the stream frame. The two tails extend to
$\phi_1 \approx \pm 6°$ inside the SDSS footprint; the curved edges are the footprint boundaries.

## 7.3 Track, width and significance along the stream

In bins of 1° in $\phi_1$ we model the distribution of masked stars in $\phi_2$ (within ±1.5° of a centre) as a stream
plus background,

$$ \lambda(\phi_2) = \epsilon(\phi_2)\,\Big[ a + b\,\phi_2 + A\, \mathcal N(\phi_2 \mid \mu, \sigma) \Big], $$

where $\epsilon$ is the fraction of each 0.05° sub-bin that SDSS observed (from the coverage grid), and fit the binned
counts with a Poisson likelihood. $TS = 2\Delta\ln\mathcal L$ compares it with the background-only model ($A = 0$).
The parameters $\mu$ and $\sigma$ are free only under the stream model, so TS does not follow $\chi^2$ (Chapter 6).
We calibrate it instead with **control strips**: the same fit centred at $\phi_2 = \pm 3.5°$, where there is no stream.
A segment is detected if its TS exceeds the largest TS of all control fits.

Only strips lying entirely inside the observed area are fitted. In a first version we allowed partly observed strips
and corrected them with the coverage fraction; two control strips that touched the edge of the data then gave
TS ≈ 60–75, because a drop in counts at an edge looks like the side of a stream. Edges and masks are a major
source of false stream detections.

In [ ]:
cov = load('pal5_coverage')
cra, cdec = (cov.ira.to_numpy() + 0.5) * 0.1, (cov.idec.to_numpy() + 0.5) * 0.1
obs = cov.n.to_numpy() > 20
allra, alldec = np.meshgrid(np.arange(222, 244, 0.1) + 0.05, np.arange(-8, 8, 0.1) + 0.05, indexing='ij')
c1, c2 = to_stream(R, cra[obs], cdec[obs])
a1, a2 = to_stream(R, allra.ravel(), alldec.ravel())

f1_edges = np.arange(-8, 8.01, 1.0)
sub = 0.05
off = np.arange(-1.5, 1.5001, sub)                                # phi2 sub-bin edges relative to the strip centre


def exposure(lo, hi, centre):
    """Observed fraction of each phi2 sub-bin for phi1 in [lo, hi), from the 0.1-deg coverage cells."""
    e = centre + off
    n_obs = np.histogram(c2[(c1 >= lo) & (c1 < hi)], e)[0]
    n_all = np.histogram(a2[(a1 >= lo) & (a1 < hi)], e)[0]
    return np.where(n_all > 0, n_obs / np.maximum(n_all, 1), 0.0)


def fit_strip(lo, hi, centre):
    """Fit stream + linear background to masked stars with phi1 in [lo, hi) and |phi2 - centre| < 1.5."""
    eps = exposure(lo, hi, centre)
    if eps.min() < 0.9:                                           # only strips fully inside the observed area
        return None
    x = 0.5 * (off[1:] + off[:-1])
    n = np.histogram(phi2[(phi1 >= lo) & (phi1 < hi)] - centre, off)[0]

    def nll(q, stream=True):
        a, b = q[0], q[1]
        lam = a + b * x
        if stream:
            A, mu, s = np.exp(q[2]), q[3], q[4]
            lam = lam + A * sub * np.exp(-0.5 * ((x - mu) / s)**2) / (np.sqrt(2 * np.pi) * s)
        lam = np.maximum(lam * eps, 1e-9)
        return -np.sum(n * np.log(lam) - lam)

    a0 = max(n[eps > 0].mean() / max(eps[eps > 0].mean(), 1e-3), 0.1)
    r0 = minimize(nll, [a0, 0.0], args=(False,), method='Nelder-Mead')
    best = None
    for mu0 in (-0.6, -0.3, 0.0, 0.3, 0.6):                       # several starting points for the stream centre
        r = minimize(nll, [*r0.x, np.log(max(a0 * 2, 1)), mu0, 0.15], method='L-BFGS-B',
                     bounds=[(1e-3, None), (None, None), (-5, 8), (-1.2, 1.2), (0.05, 0.5)])
        if best is None or r.fun < best.fun:
            best = r
    return dict(TS=2 * (r0.fun - best.fun), mu=best.x[3] + centre, sigma=best.x[4], A=np.exp(best.x[2]), n=n.sum())


rows = []
for lo, hi in zip(f1_edges[:-1], f1_edges[1:]):
    if lo < 1 and hi > -1:                                        # skip the cluster itself
        continue
    fits = {c: fit_strip(lo, hi, c) for c in (0.0, -3.5, 3.5)}
    rows.append(dict(phi1=0.5 * (lo + hi), **{f'{k}_{c}': (v[k] if v else np.nan) for c, v in fits.items() for k in ('TS', 'mu', 'sigma', 'A')}))
tr = pd.DataFrame(rows)
ctrl = np.concatenate([tr['TS_-3.5'], tr['TS_3.5']])
thresh = np.nanmax(ctrl)
tr['detected'] = tr['TS_0.0'] > thresh
print(f'largest control-strip TS: {thresh:.1f}')
print(tr[['phi1', 'TS_0.0', 'mu_0.0', 'sigma_0.0', 'A_0.0', 'detected']].round(2).to_string(index=False))

## 7.4 Comparison with a published track

`galstreams` (Mateu 2023, https://ui.adsabs.harvard.edu/abs/2023MNRAS.520.5225M) collects published stream tracks.
We take the Pal 5 track of Price-Whelan et al. (2019) from a fixed commit of the `galstreams` repository
(BSD-3 licence), transform it into our frame and compare.

In [ ]:
GALSTREAMS = ('https://raw.githubusercontent.com/cmateu/galstreams/6a304c05bea7f774390a11082ea127719e99b70a/'
              'galstreams/tracks/track.st.Pal5.pricewhelan2019.ecsv')
gs = pd.read_csv(GALSTREAMS, comment='#')
g1, g2 = to_stream(R, gs.ra.to_numpy(), gs.dec.to_numpy())
o = np.argsort(g1)
gs_phi2 = np.interp(tr.phi1, g1[o], g2[o])
det = tr[tr.detected]
resid = det['mu_0.0'] - gs_phi2[tr.detected]
print(f'detected segments: {tr.detected.sum()} ({(det.phi1 < 0).sum()} with phi1 < 0, {(det.phi1 > 0).sum()} with phi1 > 0); '
      f'median |track - galstreams| = {np.median(np.abs(resid)):.2f} deg')

fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax[0].hist2d(phi1, phi2, [np.arange(-8, 8.01, 0.1), np.arange(-2, 2.01, 0.05)], cmap='Greys', vmax=10)
ax[0].plot(g1[o], g2[o], c='C1', lw=1, label='galstreams: Price-Whelan et al. (2019)')
ax[0].errorbar(det.phi1, det['mu_0.0'], yerr=det['sigma_0.0'], fmt='o', c='C3', label='this fit: centre ± width')
ax[0].set(ylabel=r'$\phi_2$ [deg]', ylim=(-2, 2), xlim=(-8, 8))
ax[0].legend(fontsize=8)
ax[1].bar(tr.phi1, tr['TS_0.0'], width=0.8, color=np.where(tr.detected, 'C3', '0.6'), label='stream strip')
ax[1].plot(tr.phi1, tr['TS_-3.5'], 'v', c='C0', label='control strips')
ax[1].plot(tr.phi1, tr['TS_3.5'], '^', c='C0')
ax[1].axhline(thresh, c='C0', ls='--', lw=1)
ax[1].set(ylabel='TS', yscale='symlog')
ax[1].legend(fontsize=8)
ax[2].errorbar(det.phi1, det['A_0.0'], fmt='o', c='k')
ax[2].set(xlabel=r'$\phi_1$ [deg]', ylabel='stream stars per degree')
plt.tight_layout()
assert (det.phi1 < 0).sum() >= 2 and (det.phi1 > 0).sum() >= 2
assert np.median(np.abs(resid)) < 0.2

**Figure 7.2.** Top: masked stars in the stream frame, our measured track (red: centre ± Gaussian width) and the
published track (orange). Middle: TS of the stream fit in each 1° segment (red: above the largest control-strip TS,
dashed line); triangles are the control strips. Bottom: the number of stream stars per degree in the detected
segments. The tails are detected on both sides, follow the published track to a few hundredths of a degree, are
only ~0.1–0.2° wide (40–80 pc at 22 kpc), and thin out away from the cluster. The tail at $\phi_1 < 0$ leaves the SDSS
footprint at $\phi_1 \approx -3°$; segments containing small coverage holes are not fitted.

A few caveats. Our CMD mask uses one distance for the whole stream; in reality the tails span a range of distances,
so the mask loses stars at the ends. The great circle is only an approximation of the curved track (the fit absorbs
it through μ). And with only two control strips per segment, the threshold is a rough calibration: a stricter one
uses many control strips, or injected streams (Chapter 8).

## Summary

- A great-circle frame through a stream turns a curved, inclined structure into a nearly horizontal band.
- Binned Poisson fits of stream + background, including the survey footprint, give the track, width and density.
- Control strips calibrate the segment significance; the measured Pal 5 track agrees with the published one.